In [1]:
# Importing dependencies
import duckdb # instance database for sql
import pandas as pd # data handler
import json

In [20]:
!git clone https://github.com/Bcliffwm/awesome_database_project.git

Cloning into 'awesome_database_project'...
remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (7/7), done.
remote: Total 9 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (9/9), 1.33 MiB | 4.39 MiB/s, done.


EDA Step and Loading Data Into Environment

In [21]:
# Reading data from its source
my_df = pd.read_json('/content/awesome_database_project/US_STATE_recipes.json', orient='index').rename(columns={'Contient': 'Continent'})
my_df.head(5)

,Continent,Country_State,cuisine,title,URL,rating,total_time,prep_time,cook_time,description,ingredients,instructions,nutrients,serves
0,Africa,NaN,Missouri,Ground Beef and Cabbage,https://www.allrecipes.com/recipe/229324/groun...,4.5,60.0,15.0,45.0,This ground beef and cabbage recipe combines l...,"[1 large head cabbage, finely chopped, 1 (14.5...","[Place cabbage, tomatoes with juice, onion, It...","{'calories': '228 kcal', 'carbohydrateContent'...",6 servings
1,Africa,NaN,Missouri,Old Fashioned Peach Cobbler,https://www.allrecipes.com/recipe/19897/old-fa...,4.6,130.0,30.0,70.0,This old-fashioned peach cobbler recipe featur...,"[2.5 cups all-purpose flour, 4 tablespoons whi...","[Make crust: Sift together flour, 3 tablespoon...","{'calories': '338 kcal', 'carbohydrateContent'...",18 servings
2,Africa,NaN,Missouri,St. Louis Toasted Ravioli,https://www.allrecipes.com/recipe/16907/st-lou...,4.5,25.0,15.0,10.0,Toasted ravioli traces its roots to St. Louis....,"[1 (16 ounce) jar marinara sauce, 1 large egg,...",[Heat marinara sauce in a saucepan over medium...,"{'calories': '374 kcal', 'carbohydrateContent'...",6 servings
3,Africa,NaN,Missouri,Amish Friendship Bread Starter,https://www.allrecipes.com/recipe/7063/amish-f...,4.7,14440.0,30.0,NaN,Amish friendship bread starter is made with ye...,"[1 (.25 ounce) package active dry yeast, 0.25 ...",[Dissolve yeast in warm water in a small bowl;...,"{'calories': '34 kcal', 'carbohydrateContent':...",120 servings
4,Africa,NaN,Missouri,Simple Fried Morel Mushrooms,https://www.allrecipes.com/recipe/220833/simpl...,4.7,30.0,20.0,10.0,"The rich, meaty flavor of fresh morel mushroom...",[1 pound fresh morel mushrooms - dirt gently b...,[Place halved morel mushrooms in a large bowl;...,"{'calories': '185 kcal', 'carbohydrateContent'...",4 servings


In [ ]:
# Profiling global statistics of the dataset we are working with
my_df.describe()

In [ ]:
# Notice Country_State is entirely NaN - dropping any null values from the dataset
# Dropping Country_State from the dataframe
my_df = my_df.drop(columns=['Country_State'])

my_df.dropna(axis=0, inplace=True)

In [ ]:
# Understanding the dataframe's structure to form schema
my_df.info()

In [ ]:
# jsonifying the string column - nutrients
normalized_data = [] # list to hold jsonified objects

# iterating through the column
for row in my_df['nutrients']:
  normalized_data.append(pd.json_normalize(row)) # using pandas to jsonify each row's entry into a dataframe

nutrients_df = pd.concat(normalized_data, ignore_index=True) # concatenating each smaller df into one comprehense dataframe

In [ ]:
# Dropping redundant or useless columns from dataframes
nutrients_df.drop(columns=['servingSize'], inplace=True)
my_df.drop(columns=(['ingredients', 'instructions', 'nutrients', 'cuisine']), inplace=True)

In [ ]:
# Converting each record from column - serves into an int data type
my_df['serves'] = [int(entry.split(' ', 1)[0]) for entry in my_df['serves']]

In [ ]:
my_df.info()

In [ ]:
# Creating index cols for joining the two tables together
my_df['id'] = my_df.index #primary key for our recipe database
nutrients_df['recipe_id'] = nutrients_df.index # foreign key to id column of my_df table

In [ ]:
nutrients_df[0:2]

In [ ]:
# Creating an ordered column based on desired table structure for reading in database
recipe_df = my_df.loc[:, ['id', 'title', 'description', 'Continent', 'rating', 'prep_time', 'cook_time', 'serves']]

In [ ]:
nutrients_df.rename(columns={
    'carbohydrateContent': 'carbs',
    'cholesterolContent': 'cholesterol',
    'fiberContent': 'fiber',
    'proteinContent': 'protein',
    'saturatedFatContent': 'sat_fats',
    'sodiumContent': 'sodium',
    'sugarContent': 'sugar',
    'unsaturatedFatContent': 'unsat_fats'
}, inplace=True)

nutrients_df = nutrients_df.loc[:, ['recipe_id', 'calories', 'carbs', 'cholesterol', 'fiber', 'protein', 'sat_fats', 'sodium', 'sugar', 'unsat_fats']]

Initializing Instance Database for testing

In [ ]:
recipe_df.keys()

In [ ]:
# Initializing the instance database
con = duckdb.connect(database=':memory')

In [ ]:
con.execute("""CREATE TABLE recipe (id INTEGER,
  title VARCHAR(30),
  description VARCHAR(1000),
  Continent VARCHAR(30),
  rating DOUBLE,
  prep_time DOUBLE,
  cook_time DOUBLE,
  serves INTEGER
)""")

In [ ]:
con.execute("""INSERT INTO recipe (SELECT * FROM recipe_df)""")

In [ ]:
nutrients_df.keys()

In [ ]:
con.execute("""SELECT * FROM recipe""").fetchone()

In [ ]:
con.execute("""DROP TABLE IF EXISTS nutrients;
CREATE TABLE nutrients (
  recipe_id INTEGER,
  calories VARCHAR(10),
  carbs VARCHAR(10),
  cholesterol VARCHAR(10),
  fiber VARCHAR(10),
  protein VARCHAR(10),
  sat_fats VARCHAR(10),
  sodium VARCHAR(10),
  sugar VARCHAR(10),
  unsat_fats VARCHAR(10)
)""")

In [ ]:
con.execute("""INSERT INTO nutrients(SELECT * FROM nutrients_df)""")

In [ ]:
con.execute("""SELECT * FROM nutrients WHERE left(unsat_fats, 1) = '0'
LIMIT 3""").fetchdf()